# Phase 2 Demo: NYISO Energy Price Pipeline

This notebook demonstrates the Phase 2 energy price pipeline:
- Fetching NYISO LBMP data
- Storing in Parquet format
- Querying with DuckDB
- Basic analytics and visualization


In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path().resolve().parent))

import pandas as pd
import duckdb
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


## 1. Connect to DuckDB


In [ ]:
# Connect to DuckDB
db_path = Path("../analytics_db/analytics.duckdb")
parquet_base = Path("../data/parquet")

con = duckdb.connect(str(db_path))

# Refresh views
from storage.duckdb_views import DuckDBViews
views = DuckDBViews(db_path, parquet_base)
views.refresh_views()
views.close()

print("✓ Connected to DuckDB")


## 2. Query Energy Prices


In [ ]:
# Query recent day-ahead prices
query = """
SELECT 
    iso_code,
    node_id,
    zone,
    start_time,
    price
FROM energy_prices_da
WHERE start_time >= CURRENT_DATE - INTERVAL '7 days'
ORDER BY start_time DESC
LIMIT 100
"""

df = con.execute(query).df()
print(f"Retrieved {len(df)} records")
df.head()


## 3. Analyze Price Trends


In [ ]:
# Average prices by zone
query = """
SELECT 
    zone,
    AVG(price) as avg_price,
    MIN(price) as min_price,
    MAX(price) as max_price,
    COUNT(*) as record_count
FROM energy_prices_da
WHERE start_time >= CURRENT_DATE - INTERVAL '30 days'
GROUP BY zone
ORDER BY avg_price DESC
"""

df_summary = con.execute(query).df()
df_summary


## 4. Visualize Price Trends


In [ ]:
# Time series of prices for NYC zone
query = """
SELECT 
    start_time,
    price
FROM energy_prices_da
WHERE zone = 'NYC'
  AND start_time >= CURRENT_DATE - INTERVAL '7 days'
ORDER BY start_time
"""

df_nyc = con.execute(query).df()
df_nyc['start_time'] = pd.to_datetime(df_nyc['start_time'])

plt.figure(figsize=(14, 6))
plt.plot(df_nyc['start_time'], df_nyc['price'], linewidth=2)
plt.title('NYISO Day-Ahead LBMP: NYC Zone (Last 7 Days)', fontsize=14, fontweight='bold')
plt.xlabel('Time')
plt.ylabel('Price ($/MWh)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Compare Zones


In [ ]:
# Compare multiple zones
query = """
SELECT 
    start_time,
    zone,
    price
FROM energy_prices_da
WHERE zone IN ('NYC', 'LI', 'WEST', 'CENTRL')
  AND start_time >= CURRENT_DATE - INTERVAL '3 days'
ORDER BY start_time, zone
"""

df_zones = con.execute(query).df()
df_zones['start_time'] = pd.to_datetime(df_zones['start_time'])

plt.figure(figsize=(14, 6))
for zone in df_zones['zone'].unique():
    zone_data = df_zones[df_zones['zone'] == zone]
    plt.plot(zone_data['start_time'], zone_data['price'], label=zone, linewidth=2)

plt.title('NYISO Day-Ahead LBMP: Zone Comparison (Last 3 Days)', fontsize=14, fontweight='bold')
plt.xlabel('Time')
plt.ylabel('Price ($/MWh)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Data Quality Check


In [ ]:
# Check data coverage
query = """
SELECT 
    iso_code,
    COUNT(DISTINCT zone) as zone_count,
    COUNT(*) as total_records,
    MIN(start_time) as earliest_date,
    MAX(start_time) as latest_date
FROM energy_prices_da
GROUP BY iso_code
"""

df_coverage = con.execute(query).df()
df_coverage


## 7. Check Ingestion Log


In [ ]:
import sqlite3

# Connect to metadata DB
metadata_db = Path("../metadata/metadata.db")
conn = sqlite3.connect(str(metadata_db))

query = """
SELECT 
    dataset_name,
    iso_code,
    start_time,
    end_time,
    status,
    row_count
FROM ingestion_log
ORDER BY start_time DESC
LIMIT 10
"""

df_log = pd.read_sql_query(query, conn)
conn.close()

df_log


## Summary

This demo shows:
- ✓ Data ingestion from NYISO
- ✓ Storage in Parquet format
- ✓ Querying with DuckDB
- ✓ Basic analytics and visualization

Next steps:
- Add more ISOs (PJM, ISO-NE, etc.)
- Add capacity market data (Phase 3)
- Add semantic search with Qdrant (Phase 5)
